In [2]:
# ASSESSMENT CELL 1 — Setup: install, connect Drive, load tools
!pip install -q pillow-heif

from google.colab import drive
drive.mount('/content/drive')

import os, json
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from PIL import Image, ImageOps
from pillow_heif import register_heif_opener

register_heif_opener()   # lets us open iPhone photos (.HEIC)
print("Setup done. TensorFlow version:", tf.__version__)

Mounted at /content/drive
Setup done. TensorFlow version: 2.21.0


In [3]:
# ASSESSMENT CELL 2 — Load the saved model and the prediction tools
PROJECT = '/content/drive/MyDrive/capstone'

model = keras.models.load_model(os.path.join(PROJECT, 'model', 'cat_dog_neither.keras'))
with open(os.path.join(PROJECT, 'model', 'class_names.json')) as f:
    CLASS_NAMES = json.load(f)          # ['dog', 'cat', 'neither']

IMG_SIZE = 224
IMAGE_TYPES = ('.jpg', '.jpeg', '.png', '.webp', '.bmp', '.heic', '.heif')

def load_photo(path):
    """Open any photo file and return it the right way up, in normal colour."""
    img = Image.open(path)
    img = ImageOps.exif_transpose(img)   # fix sideways phone photos
    img = img.convert('RGB')             # fix PNG transparency / black-and-white
    return np.array(img)

def resize_for_model(image):
    """Resize to 224x224 without stretching - EXACTLY the same as in training."""
    return tf.image.resize_with_pad(tf.cast(image, tf.float32), IMG_SIZE, IMG_SIZE)

def predict_folder(folder):
    """Run the model on every photo in a folder. Return one row per photo."""
    rows = []
    for filename in sorted(os.listdir(folder)):
        path = os.path.join(folder, filename)
        if not os.path.isfile(path) or not filename.lower().endswith(IMAGE_TYPES):
            continue                                         # skip folders and non-photos
        try:
            image = resize_for_model(load_photo(path))
            probs = model.predict(tf.expand_dims(image, 0), verbose=0)[0]
            best = int(np.argmax(probs))
            rows.append({'filename': filename,
                         'prediction': CLASS_NAMES[best],
                         'confidence': f"{probs[best] * 100:.1f}%"})
        except Exception as error:                           # broken file: report it, don't crash
            rows.append({'filename': filename, 'prediction': 'ERROR',
                         'confidence': str(error)[:40]})
    return pd.DataFrame(rows, columns=['filename', 'prediction', 'confidence'])

# Warm-up: the first prediction is always slow, so do it now rather than live in class
model.predict(tf.zeros((1, IMG_SIZE, IMG_SIZE, 3)), verbose=0)
print("Model loaded. Classes:", CLASS_NAMES)
print("Ready to predict.")

Model loaded. Classes: ['dog', 'cat', 'neither']
Ready to predict.


In [5]:
# ASSESSMENT CELL 3 — Change ONLY the folder path below, then run
FOLDER = '/content/drive/MyDrive/capstone/demo_photos'

os.makedirs(FOLDER, exist_ok=True)     # creates the folder if it doesn't exist yet
results = predict_folder(FOLDER)

if results.empty:
    print("No photos found in", FOLDER, "- add some and run this cell again.")
else:
    print(results.to_string(index=False))
    print("\nTotals:", results['prediction'].value_counts().to_dict())

                                         filename prediction confidence
WhatsApp Image 2026-11-16 at 12.52.08 PM (1).jpeg    neither      80.5%
WhatsApp Image 2026-11-16 at 12.52.08 PM (2).jpeg    neither      59.3%
    WhatsApp Image 2026-11-16 at 12.52.08 PM.jpeg    neither      64.3%
WhatsApp Image 2026-11-16 at 12.52.09 PM (1).jpeg    neither      81.7%
WhatsApp Image 2026-11-16 at 12.52.09 PM (2).jpeg    neither      72.9%
    WhatsApp Image 2026-11-16 at 12.52.09 PM.jpeg    neither      97.8%

Totals: {'neither': 6}
